# DL 시퀀스 데이터 준비
Purpose: build deterministic causal sequence indexes from the verified Goal 1.5 ML role views.

> Warning: this is an oracle/sanity-only synthetic-data benchmark, not real-device or medical-performance evidence.

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd

SERIES_ID = "mvp3-oracle-v1"
EXPECTED_SPLIT_COUNTS = {"train": 24, "validation": 6, "locked_test": 6}
DATA_STATUS = "oracle/sanity"
REAL_ACCURACY_STATUS = "NOT VERIFIED"
DEVICE_SYNCHRONIZATION_STATUS = "NOT_AVAILABLE_TRUTH_ONLY"
RUN_TRAINING = False
RUN_LOCKED_TEST = False
SEQUENCE_LENGTHS_SECONDS = (300, 600)
SEQUENCE_OUTPUT_ROOT = Path("/kaggle/working/goal15_dl_sequences")
ML_VIEW_ROOT = Path("/kaggle/working/goal15_ml_view")
PATTERN_TARGET = "pattern_binary"
ONSET_EVENT_TARGET = "event_binary"
STAGE_TARGET = "stage_code"
BEHAVIOR_CODES = (
    "ear_covering", "exit_attempt", "head_turn_away", "motion_freeze",
    "movement_reduction", "repetitive_body_movement",
    "repetitive_hand_movement", "repetitive_object_contact",
    "sustained_pressure_or_contact", "withdrawal_movement",
)
CAUSAL_FACTORS = (
    "autonomic_arousal", "motor_activation", "cognitive_load", "sleep_pressure",
    "sensory_context", "recovery_capacity", "social_context",
)
ROLLING_STATISTICS = ("mean", "std", "slope")
ROLLING_WINDOWS_SECONDS = (5, 15, 30, 60, 180, 300)
TIME_FEATURE_COLUMNS = ("time_sin", "time_cos", "weekday_sin", "weekday_cos", "is_awake")
CONTEXT_FEATURE_COLUMNS = (
    "context__sleep", "context__transition", "context__meal_context",
    "context__focused_task", "context__moderate_activity",
    "context__light_activity", "context__wake_rest",
    "context__sedentary_activity",
)
ALLOWED_FEATURE_COLUMNS = tuple(
    [
        feature
        for factor in CAUSAL_FACTORS
        for feature in (
            f"{factor}__robust_z",
            *(
                f"{factor}__{statistic}_{window_seconds}s"
                for window_seconds in ROLLING_WINDOWS_SECONDS
                for statistic in ROLLING_STATISTICS
            ),
        )
    ]
    + list(TIME_FEATURE_COLUMNS)
    + list(CONTEXT_FEATURE_COLUMNS)
)


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _require_sha256(value: Any, field: str) -> str:
    if not isinstance(value, str) or re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"invalid {field} hash")
    return value


def validate_shared_dataset_identity(manifest: Mapping[str, Any]) -> tuple[str, str]:
    if manifest.get("series_id") != SERIES_ID:
        raise ValueError("unexpected series identity")
    if manifest.get("data_status") != DATA_STATUS:
        raise ValueError("unexpected data status")
    source_dataset_hash = _require_sha256(manifest.get("source_dataset_hash"), "source dataset")
    split_hash = _require_sha256(manifest.get("split_hash"), "split")
    return source_dataset_hash, split_hash


def fit_train_normalization(
    frame: pd.DataFrame, *, feature_columns: Sequence[str], source_hash: str
) -> dict[str, Any]:
    _require_sha256(source_hash, "source")
    if "split_role" not in frame:
        raise ValueError("normalization requires split_role")
    missing = sorted(set(feature_columns).difference(frame.columns))
    if missing:
        raise ValueError(f"normalization features missing: {missing}")
    train = frame.loc[frame["split_role"].eq("train"), list(feature_columns)]
    if train.empty:
        raise ValueError("normalization requires train rows")
    statistics: dict[str, dict[str, float]] = {}
    for feature in feature_columns:
        values = pd.to_numeric(train[feature], errors="raise").to_numpy(dtype=np.float64)
        if not np.isfinite(values).all():
            raise ValueError(f"non-finite train feature: {feature}")
        median = float(np.median(values))
        lower, upper = np.quantile(values, [0.25, 0.75])
        iqr = float(upper - lower)
        statistics[feature] = {"median": median, "iqr": iqr if iqr > 0 else 1.0}
    return {
        "series_id": SERIES_ID,
        "fit_split_role": "train",
        "source_hash": source_hash,
        "features": statistics,
    }


def _window_group_keys(frame: pd.DataFrame) -> list[str]:
    required = ["person_key", "run_id", "dataset_id", "canonical_time"]
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        raise ValueError(f"sequence source missing columns: {missing}")
    group_keys = ["person_key", "run_id", "dataset_id"]
    group_keys.extend(key for key in ("day_key", "day", "date", "session_id") if key in frame)
    return group_keys


def assert_window_boundaries(index: pd.DataFrame) -> None:
    required = {"person_key", "run_id", "dataset_id", "window_start", "window_end", "prediction_time", "length_seconds"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"sequence index missing columns: {missing}")
    # Every candidate must satisfy window_end >= window_start.
    if not (index["window_end"] >= index["window_start"]).all():
        raise ValueError("window_end must be after window_start")
    expected_end = index["prediction_time"]
    if not index["window_end"].eq(expected_end).all():
        raise ValueError("window_end must equal prediction_time")
    expected_start = expected_end - pd.to_timedelta(index["length_seconds"] - 1, unit="s")
    if not index["window_start"].eq(expected_start).all():
        raise ValueError("causal window start mismatch")


In [ ]:
def make_causal_window_index(frame: pd.DataFrame, *, length_seconds: int) -> pd.DataFrame:
    if isinstance(length_seconds, bool) or length_seconds < 1:
        raise ValueError("length_seconds must be positive")
    group_keys = _window_group_keys(frame)
    work = frame.copy()
    work["canonical_time"] = pd.to_datetime(work["canonical_time"], utc=True)
    if work["canonical_time"].isna().any():
        raise ValueError("canonical_time must be complete")
    missing_column = next((name for name in ("missing_block", "is_missing_block") if name in work), None)
    records: list[dict[str, Any]] = []
    for _, group in work.groupby(group_keys, sort=False, dropna=False):
        ordered = group.sort_values("canonical_time", kind="mergesort").reset_index(drop=True)
        if ordered["canonical_time"].duplicated().any():
            raise ValueError("duplicate canonical_time within sequence group")
        for end_index in range(length_seconds - 1, len(ordered)):
            window = ordered.iloc[end_index - length_seconds + 1 : end_index + 1]
            prediction_time = window["canonical_time"].iloc[-1]
            window_start = prediction_time - pd.Timedelta(seconds=length_seconds - 1)
            window_end = prediction_time
            consecutive = window["canonical_time"].diff().dropna().eq(pd.Timedelta(seconds=1)).all()
            has_missing_block = bool(window[missing_column].fillna(True).astype(bool).any()) if missing_column else False
            if not consecutive or has_missing_block:
                continue
            record = window.iloc[-1].to_dict()
            record.update({
                "window_start": window_start,
                "window_end": window_end,
                "prediction_time": prediction_time,
                "length_seconds": length_seconds,
            })
            record["window_id"] = hashlib.sha256(
                "|".join(str(record[key]) for key in (*group_keys, "prediction_time", "length_seconds")).encode()
            ).hexdigest()
            records.append(record)
    columns = [*frame.columns, "window_start", "window_end", "prediction_time", "length_seconds", "window_id"]
    index = pd.DataFrame(records, columns=list(dict.fromkeys(columns)))
    if not index.empty:
        assert_window_boundaries(index)
    return index


def sample_training_windows(index: pd.DataFrame, *, baseline_multiplier: int = 3) -> pd.DataFrame:
    if baseline_multiplier < 0:
        raise ValueError("baseline_multiplier must be non-negative")
    required = {"split_role", PATTERN_TARGET, "hard_negative", "window_id"}
    missing = sorted(required.difference(index.columns))
    if missing:
        raise ValueError(f"training index missing columns: {missing}")
    if not index["split_role"].eq("train").all():
        raise ValueError("training sampler accepts train windows only")
    positive = index[PATTERN_TARGET].eq(1)
    hard_negative = ~positive & index["hard_negative"].eq(1)
    baseline = ~(positive | hard_negative)
    selected = [
        index.loc[positive].assign(sample_type="positive_centered"),
        index.loc[hard_negative].assign(sample_type="hard_negative"),
    ]
    candidates = index.loc[baseline].copy()
    candidates["_sample_hash"] = pd.util.hash_pandas_object(
        candidates[["person_key", "run_id", "dataset_id", "prediction_time", "window_id"]],
        index=False,
        categorize=True,
    )
    selected.append(
        candidates.sort_values(["_sample_hash", "window_id"], kind="mergesort")
        .head(baseline_multiplier * int(positive.sum()))
        .drop(columns="_sample_hash")
        .assign(sample_type="matched_baseline")
    )
    return pd.concat(selected, ignore_index=True).sort_values(
        ["prediction_time", "window_id"], kind="mergesort"
    ).reset_index(drop=True)


def write_train_normalization(output_root: Path, statistics: Mapping[str, Any]) -> Path:
    output_root.mkdir(parents=True, exist_ok=True)
    path = output_root / "train_normalization.json"
    path.write_text(json.dumps(statistics, indent=2, sort_keys=True) + "\n")
    return path


def write_sequence_manifest(
    output_root: Path, *, index_paths: Mapping[str, Path], normalization_path: Path,
    source_dataset_hash: str, split_hash: str, row_counts: Mapping[str, int],
) -> Path:
    _require_sha256(source_dataset_hash, "source dataset")
    _require_sha256(split_hash, "split")
    if set(index_paths) != set(row_counts):
        raise ValueError("sequence manifest role metadata mismatch")
    if not normalization_path.is_file():
        raise FileNotFoundError("missing train normalization statistics")
    files: dict[str, dict[str, Any]] = {}
    for name, path in sorted(index_paths.items()):
        count = row_counts[name]
        if not path.is_file() or isinstance(count, bool) or not isinstance(count, int) or count < 0:
            raise ValueError(f"invalid sequence output: {name}")
        files[name] = {"path": path.name, "sha256": sha256_file(path), "row_count": count}
    manifest_path = output_root / "sequence_manifest.json"
    manifest_path.write_text(json.dumps({
        "series_id": SERIES_ID, "data_status": DATA_STATUS,
        "source_dataset_hash": source_dataset_hash, "split_hash": split_hash,
        "normalization": {"path": normalization_path.name, "sha256": sha256_file(normalization_path)},
        "files": files,
    }, indent=2, sort_keys=True) + "\n")
    return manifest_path


In [ ]:
def resolve_ml_view_root(input_root: Path = Path("/kaggle/input")) -> Path:
    candidates = [ML_VIEW_ROOT, input_root, *sorted(path for path in input_root.iterdir() if path.is_dir())]
    for candidate in candidates:
        if (candidate / "view_manifest.json").is_file():
            return candidate
    raise FileNotFoundError("verified goal15_ml_view manifest is required")


def _verified_role_views(root: Path) -> tuple[dict[str, pd.DataFrame], str, str]:
    manifest = json.loads((root / "view_manifest.json").read_text())
    source_dataset_hash, split_hash = validate_shared_dataset_identity(manifest)
    files = manifest.get("files")
    if not isinstance(files, dict) or set(files) != set(EXPECTED_SPLIT_COUNTS):
        raise ValueError("ML view manifest must declare all immutable split roles")
    views: dict[str, pd.DataFrame] = {}
    people: dict[str, set[str]] = {}
    for split_role, metadata in files.items():
        path = root / metadata["path"]
        if not path.is_file() or sha256_file(path) != _require_sha256(metadata.get("sha256"), split_role):
            raise ValueError(f"unverified ML role view: {split_role}")
        frame = pd.read_parquet(path).copy()
        frame["split_role"] = split_role
        required = {"person_key", "run_id", "dataset_id", "canonical_time", PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", STAGE_TARGET, *BEHAVIOR_CODES}
        missing = sorted(required.difference(frame.columns))
        if missing:
            raise ValueError(f"ML role view missing contract columns: {missing}")
        views[split_role] = frame
        people[split_role] = set(frame["person_key"].astype(str))
    if {role: len(people[role]) for role in EXPECTED_SPLIT_COUNTS} != EXPECTED_SPLIT_COUNTS:
        raise ValueError("ML role views do not preserve 24/6/6 people")
    if any(people[left] & people[right] for left in people for right in people if left < right):
        raise ValueError("person leakage across DL sequence roles")
    return views, source_dataset_hash, split_hash


def _select_causal_features(frame: pd.DataFrame) -> list[str]:
    unexpected = sorted(set(frame.columns).difference(ALLOWED_FEATURE_COLUMNS) - {
        "person_key", "person_id", "run_id", "dataset_id", "canonical_time",
        "split_role", PATTERN_TARGET, ONSET_EVENT_TARGET, "hard_negative", STAGE_TARGET, *BEHAVIOR_CODES,
        "event_id", "session_id", "day_key", "day", "date", "missing_block", "is_missing_block",
    })
    if unexpected:
        raise ValueError(f"unapproved DL sequence columns: {unexpected}")
    features = [feature for feature in ALLOWED_FEATURE_COLUMNS if feature in frame]
    if not features:
        raise ValueError("no approved causal features in ML view")
    return features


def build_all_sequence_indexes() -> Path:
    views, source_dataset_hash, split_hash = _verified_role_views(resolve_ml_view_root())
    combined = pd.concat(views.values(), ignore_index=True)
    feature_columns = _select_causal_features(combined)
    statistics = fit_train_normalization(combined, feature_columns=feature_columns, source_hash=source_dataset_hash)
    normalization_path = write_train_normalization(SEQUENCE_OUTPUT_ROOT, statistics)
    index_paths: dict[str, Path] = {}
    row_counts: dict[str, int] = {}
    for split_role, frame in views.items():
        for length_seconds in SEQUENCE_LENGTHS_SECONDS:
            index = make_causal_window_index(frame, length_seconds=length_seconds)
            if split_role == "train":
                index = sample_training_windows(index)
            name = f"{split_role}_{length_seconds}"
            path = SEQUENCE_OUTPUT_ROOT / f"{name}.parquet"
            index.to_parquet(path, index=False, compression="zstd")
            index_paths[name] = path
            row_counts[name] = len(index)
    return write_sequence_manifest(
        SEQUENCE_OUTPUT_ROOT, index_paths=index_paths, normalization_path=normalization_path,
        source_dataset_hash=source_dataset_hash, split_hash=split_hash, row_counts=row_counts,
    )


In [ ]:
RUN_DATA_PREPARATION = False

if RUN_DATA_PREPARATION:
    manifest_path = build_all_sequence_indexes()
    print(f"시퀀스 인덱스 생성 완료: {manifest_path}")
else:
    print("시퀀스 데이터 준비 비활성화: RUN_DATA_PREPARATION=False")
